# Day 065 — Solution: Capstone Build II

In [ ]:
_SHIPPED_SRC  = '"""shipped_app.py — Day 065: Capstone Build II — shippable AI Writing Assistant.\n\nAll Day 064 features + monitoring middleware + /metrics endpoint.\n\nSetup:\n  pip install fastapi "uvicorn[standard]" ollama\n  ollama pull llama3.2\n\nRun:  uvicorn shipped_app:app --reload\nDocs: http://localhost:8000/docs\nTest: pytest test_shipped_app.py -v\n"""\nimport os\nimport re\nimport secrets\nimport time\nfrom datetime import datetime\n\nimport ollama\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nAPP_VER = "1.1.0"\nMODEL   = os.environ.get("MODEL", "llama3.2")\n\n# ── plan configuration ────────────────────────────────────────────────────────\n\nDAILY_LIMITS = {"free": 5, "pro": 500, "enterprise": float("inf")}\n\nFEATURE_MATRIX = {\n    "free": {"basic_generate", "view_history"},\n    "pro":  {"basic_generate", "view_history", "improve_text", "export"},\n}\n\nTEMPLATES = {\n    "email":      "Write a {tone} email to {recipient} about {topic}.",\n    "tweet":      "Write a {tone} tweet about {topic} in under 280 characters.",\n    "summary":    "Write a concise {length}-sentence summary of: {content}",\n    "blog_intro": "Write an engaging blog intro about {topic} for a {audience} audience.",\n}\n\n\ndef check_feature_access(plan: str, feature: str) -> bool:\n    return feature in FEATURE_MATRIX.get(plan, set())\n\n\ndef check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:\n    limit = DAILY_LIMITS.get(plan, 0)\n    if usage_count >= limit:\n        return False, (\n            f"Daily limit reached for {plan!r} plan "\n            f"({usage_count}/{int(limit)}). Upgrade to pro for 500/day."\n        )\n    return True, ""\n\n\ndef render_template(template_str: str, **vars) -> str:\n    required = set(re.findall(r\'\\{(\\w+)\\}\', template_str))\n    missing  = required - set(vars.keys())\n    if missing:\n        raise ValueError(f"Missing template variables: {missing}")\n    result = template_str\n    for key, value in vars.items():\n        result = result.replace(f"{{{key}}}", str(value))\n    return result\n\n\n# ── storage ───────────────────────────────────────────────────────────────────\n\nclass ContentStore:\n    def __init__(self):\n        self._store: dict = {}\n\n    def add(self, user_id: str, prompt: str, content: str) -> str:\n        cid = secrets.token_urlsafe(8)\n        self._store[cid] = {\n            "content_id": cid, "user_id": user_id,\n            "prompt": prompt, "content": content,\n            "created_at": datetime.utcnow().isoformat() + "Z",\n        }\n        return cid\n\n    def get(self, content_id: str) -> dict | None:\n        return self._store.get(content_id)\n\n    def list_user(self, user_id: str) -> list[dict]:\n        return [v for v in self._store.values() if v["user_id"] == user_id]\n\n    def count(self, user_id: str) -> int:\n        return sum(1 for v in self._store.values() if v["user_id"] == user_id)\n\n\n# ── metrics ───────────────────────────────────────────────────────────────────\n\nclass MetricsCollector:\n    def __init__(self):\n        self._requests  = 0\n        self._errors    = 0\n        self._latencies: list[float] = []\n\n    def record(self, status_code: int, duration_ms: float) -> None:\n        self._requests += 1\n        if status_code >= 400:\n            self._errors += 1\n        self._latencies.append(duration_ms)\n\n    def summary(self) -> dict:\n        avg  = (sum(self._latencies) / len(self._latencies)\n                if self._latencies else 0.0)\n        rate = self._errors / self._requests if self._requests else 0.0\n        return {\n            "requests":       self._requests,\n            "errors":         self._errors,\n            "avg_latency_ms": round(avg, 1),\n            "error_rate":     round(rate, 3),\n        }\n\n\n# ── FastAPI app ───────────────────────────────────────────────────────────────\n\ndef build_api(process_fn=None, initial_plan: str = "free",\n              initial_usage: int = 0) -> FastAPI:\n    app       = FastAPI(title="AI Writing Assistant", version=APP_VER)\n    store     = ContentStore()\n    collector = MetricsCollector()\n    state     = {"plan": initial_plan, "usage": initial_usage}\n\n    @app.middleware("http")\n    async def metrics_middleware(request, call_next):\n        start    = time.monotonic()\n        response = await call_next(request)\n        duration = (time.monotonic() - start) * 1000\n        collector.record(response.status_code, duration)\n        return response\n\n    class _GenReq(BaseModel):\n        prompt:  str = Field(min_length=1)\n        user_id: str = Field(min_length=1)\n\n    class _TplReq(BaseModel):\n        template: str = Field(min_length=1)\n        vars:     dict = {}\n        user_id:  str  = Field(min_length=1)\n\n    class _ImpReq(BaseModel):\n        text:    str = Field(min_length=1)\n        user_id: str = Field(min_length=1)\n\n    @app.get("/health")\n    def health():\n        return {"status": "ok",\n                "timestamp": datetime.utcnow().isoformat() + "Z",\n                "version": APP_VER}\n\n    @app.get("/plan")\n    def get_plan():\n        lim = DAILY_LIMITS.get(state["plan"], 0)\n        return {"plan": state["plan"], "usage_today": state["usage"],\n                "limit": lim if lim != float("inf") else -1}\n\n    @app.get("/metrics")\n    def get_metrics():\n        return collector.summary()\n\n    @app.get("/templates")\n    def list_templates():\n        return {"templates": list(TEMPLATES.keys())}\n\n    @app.post("/generate")\n    def generate(req: _GenReq):\n        ok, reason = check_rate_limit(state["usage"], state["plan"])\n        if not ok:\n            raise HTTPException(429, reason)\n        answer = (process_fn(req.prompt) if process_fn\n                  else ollama.chat(model=MODEL,\n                                   messages=[{"role": "user",\n                                              "content": req.prompt}])\n                  ["message"]["content"])\n        state["usage"] += 1\n        cid = store.add(req.user_id, req.prompt, answer)\n        return {"content_id": cid, "content": answer, "user_id": req.user_id}\n\n    @app.post("/generate/template")\n    def gen_template(req: _TplReq):\n        ok, reason = check_rate_limit(state["usage"], state["plan"])\n        if not ok:\n            raise HTTPException(429, reason)\n        tmpl = TEMPLATES.get(req.template)\n        if tmpl is None:\n            raise HTTPException(400, f"Unknown template: {req.template!r}")\n        try:\n            prompt = render_template(tmpl, **req.vars)\n        except ValueError as e:\n            raise HTTPException(400, str(e))\n        answer = (process_fn(prompt) if process_fn\n                  else ollama.chat(model=MODEL,\n                                   messages=[{"role": "user",\n                                              "content": prompt}])\n                  ["message"]["content"])\n        state["usage"] += 1\n        cid = store.add(req.user_id, prompt, answer)\n        return {"content_id": cid, "content": answer,\n                "template": req.template, "user_id": req.user_id}\n\n    @app.post("/improve")\n    def improve(req: _ImpReq):\n        if not check_feature_access(state["plan"], "improve_text"):\n            raise HTTPException(403, "improve_text requires pro plan. "\n                                     "Upgrade at /checkout")\n        ok, reason = check_rate_limit(state["usage"], state["plan"])\n        if not ok:\n            raise HTTPException(429, reason)\n        prompt = f"Improve this text for clarity and style:\\n\\n{req.text}"\n        answer = (process_fn(prompt) if process_fn\n                  else ollama.chat(model=MODEL,\n                                   messages=[{"role": "user",\n                                              "content": prompt}])\n                  ["message"]["content"])\n        state["usage"] += 1\n        cid = store.add(req.user_id, prompt, answer)\n        return {"content_id": cid, "original": req.text,\n                "improved": answer, "user_id": req.user_id}\n\n    @app.get("/history/{user_id}")\n    def history(user_id: str):\n        items = store.list_user(user_id)\n        return {"user_id": user_id, "count": len(items), "items": items}\n\n    @app.get("/content/{content_id}")\n    def get_content(content_id: str):\n        item = store.get(content_id)\n        if item is None:\n            raise HTTPException(404, f"Content {content_id!r} not found")\n        return item\n\n    return app\n\n\napp = build_api()\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
_TEST_SRC     = '"""test_shipped_app.py — pytest test suite for shipped_app.\n\nRun:\n  pytest test_shipped_app.py -v\n  pytest test_shipped_app.py -v -k generate\n"""\nimport pytest\nfrom starlette.testclient import TestClient\nfrom shipped_app import build_api\n\n\n@pytest.fixture\ndef client():\n    return TestClient(build_api(plan="free", process_fn=str.upper),\n                      raise_server_exceptions=False)\n\n\n@pytest.fixture\ndef pro_client():\n    return TestClient(build_api(plan="pro", process_fn=str.upper),\n                      raise_server_exceptions=False)\n\n\n@pytest.fixture\ndef at_limit_client():\n    return TestClient(\n        build_api(plan="free", process_fn=str.upper, initial_usage=5),\n        raise_server_exceptions=False,\n    )\n\n\n# ── infrastructure ─────────────────────────────────────────────────────────────\n\ndef test_health(client):\n    r = client.get("/health")\n    assert r.status_code == 200\n    d = r.json()\n    assert d["status"] == "ok"\n    assert "timestamp" in d and "version" in d\n\n\ndef test_plan(client):\n    r = client.get("/plan")\n    assert r.status_code == 200\n    d = r.json()\n    assert d["plan"] == "free"\n    assert d["usage_today"] == 0\n    assert d["limit"] == 5\n\n\ndef test_templates(client):\n    r = client.get("/templates")\n    assert r.status_code == 200\n    assert isinstance(r.json()["templates"], list)\n    assert len(r.json()["templates"]) > 0\n\n\ndef test_metrics(client):\n    r = client.get("/metrics")\n    assert r.status_code == 200\n    d = r.json()\n    for key in ("requests", "errors", "avg_latency_ms", "error_rate"):\n        assert key in d, f"Missing key: {key}"\n\n\n# ── generation ─────────────────────────────────────────────────────────────────\n\ndef test_generate_ok(client):\n    r = client.post("/generate", json={"prompt": "hello", "user_id": "u1"})\n    assert r.status_code == 200\n    d = r.json()\n    assert "content_id" in d\n    assert d["content"] == "HELLO"\n    assert d["user_id"] == "u1"\n\n\n@pytest.mark.parametrize("body,expected", [\n    ({"prompt": "", "user_id": "u1"}, 422),    # empty prompt\n    ({"prompt": "hi"}, 422),                   # missing user_id\n    ({}, 422),                                  # empty body\n])\ndef test_generate_validation(client, body, expected):\n    r = client.post("/generate", json=body)\n    assert r.status_code == expected, f"got {r.status_code}: {r.text[:120]}"\n\n\ndef test_generate_rate_limit(at_limit_client):\n    r = at_limit_client.post("/generate",\n                             json={"prompt": "hi", "user_id": "u1"})\n    assert r.status_code == 429\n\n\ndef test_generate_usage_increments(client):\n    client.post("/generate", json={"prompt": "a", "user_id": "u1"})\n    client.post("/generate", json={"prompt": "b", "user_id": "u1"})\n    r = client.get("/plan")\n    assert r.json()["usage_today"] == 2\n\n\n# ── history ────────────────────────────────────────────────────────────────────\n\ndef test_history_empty(client):\n    r = client.get("/history/nobody")\n    assert r.status_code == 200\n    assert r.json()["count"] == 0\n\n\ndef test_history_after_generate(client):\n    client.post("/generate", json={"prompt": "x", "user_id": "alice"})\n    client.post("/generate", json={"prompt": "y", "user_id": "alice"})\n    client.post("/generate", json={"prompt": "z", "user_id": "bob"})\n    r = client.get("/history/alice")\n    assert r.json()["count"] == 2\n    assert all(i["user_id"] == "alice" for i in r.json()["items"])\n\n\ndef test_content_get(client):\n    rg  = client.post("/generate", json={"prompt": "hi", "user_id": "u"})\n    cid = rg.json()["content_id"]\n    rc  = client.get(f"/content/{cid}")\n    assert rc.status_code == 200\n    assert rc.json()["content_id"] == cid\n\n\ndef test_content_not_found(client):\n    assert client.get("/content/does_not_exist").status_code == 404\n\n\n# ── feature gating ─────────────────────────────────────────────────────────────\n\ndef test_improve_requires_pro(client):\n    r = client.post("/improve", json={"text": "hello", "user_id": "u1"})\n    assert r.status_code == 403\n\n\ndef test_improve_works_for_pro(pro_client):\n    r = pro_client.post("/improve", json={"text": "hello", "user_id": "u1"})\n    assert r.status_code == 200\n    assert "improved" in r.json()\n\n\n# ── state isolation ────────────────────────────────────────────────────────────\n\ndef test_isolation_between_clients():\n    c1 = TestClient(build_api(plan="free", process_fn=str.upper),\n                    raise_server_exceptions=False)\n    c2 = TestClient(build_api(plan="free", process_fn=str.upper),\n                    raise_server_exceptions=False)\n    c1.post("/generate", json={"prompt": "a", "user_id": "u"})\n    assert c2.get("/history/u").json()["count"] == 0\n'
_RENDER_YAML  = 'services:\n  - type: web\n    name: writing-assistant\n    runtime: python\n    buildCommand: pip install -r requirements.txt\n    startCommand: uvicorn shipped_app:app --host 0.0.0.0 --port $PORT\n    envVars:\n      - key: MODEL\n        value: llama3.2\n      - key: STRIPE_SECRET_KEY\n        sync: false\n      - key: STRIPE_WEBHOOK_SECRET\n        sync: false\n      - key: PORT\n        generateValue: false\n'
_ENV_EXAMPLE  = '# Copy to .env and fill in values before running locally.\n# Never commit .env to git.\nMODEL=llama3.2\nPORT=8000\nSTRIPE_SECRET_KEY=sk_test_...\nSTRIPE_WEBHOOK_SECRET=whsec_...\n'
_REQUIREMENTS = 'fastapi\nhttpx\nollama\nuvicorn[standard]\n'
_PROCFILE     = 'web: uvicorn shipped_app:app --host 0.0.0.0 --port $PORT\n'
from pathlib import Path
Path('shipped_app.py').write_text(_SHIPPED_SRC)
Path('test_shipped_app.py').write_text(_TEST_SRC)
Path('render.yaml').write_text(_RENDER_YAML)
Path('.env.example').write_text(_ENV_EXAMPLE)
Path('requirements.txt').write_text(_REQUIREMENTS)
Path('Procfile').write_text(_PROCFILE)
print('All deliverable files written.')

In [ ]:
# inline integration test — no Ollama needed
import re, secrets, time
from datetime import datetime
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

DAILY_LIMITS = {"free": 5, "pro": 500}
TEMPLATES    = {"email": "Write a {tone} email about {topic}.",
                "tweet": "Write a {tone} tweet about {topic}."}

def check_rate_limit(u, p):
    lim = DAILY_LIMITS.get(p, 0)
    return (False, "limit") if u >= lim else (True, "")

class ContentStore:
    def __init__(self): self._s = {}
    def add(self, uid, p, c):
        cid = secrets.token_urlsafe(8)
        self._s[cid] = {"content_id":cid,"user_id":uid,"prompt":p,"content":c}
        return cid
    def get(self, cid): return self._s.get(cid)
    def list_user(self, uid): return [v for v in self._s.values() if v["user_id"]==uid]

class MetricsCollector:
    def __init__(self): self._req=0; self._err=0; self._lat=[]
    def record(self, sc, ms): self._req+=1; self._err+=(1 if sc>=400 else 0); self._lat.append(ms)
    def summary(self): avg=sum(self._lat)/len(self._lat) if self._lat else 0.0;         return {"requests":self._req,"errors":self._err,
                "avg_latency_ms":round(avg,1),"error_rate":round(self._err/self._req if self._req else 0,3)}

def build_core(plan="free", process_fn=None, initial_usage=0):
    app=FastAPI(); store=ContentStore(); collector=MetricsCollector()
    state={"plan":plan,"usage":initial_usage}
    class _G(BaseModel): prompt:str=Field(min_length=1); user_id:str=Field(min_length=1)
    @app.middleware("http")
    async def mw(req,call_next):
        s=time.monotonic(); r=await call_next(req)
        collector.record(r.status_code,(time.monotonic()-s)*1000); return r
    @app.get("/health")
    def h(): return {"status":"ok","version":"1.1.0"}
    @app.get("/plan")
    def p(): lim=DAILY_LIMITS.get(state["plan"],0); return {"plan":state["plan"],"usage_today":state["usage"],"limit":lim}
    @app.get("/metrics")
    def m(): return collector.summary()
    @app.get("/templates")
    def t(): return {"templates":list(TEMPLATES.keys())}
    @app.post("/generate")
    def g(req:_G):
        ok,reason=check_rate_limit(state["usage"],state["plan"])
        if not ok: raise HTTPException(429,reason)
        ans=process_fn(req.prompt) if process_fn else req.prompt.upper()
        state["usage"]+=1; cid=store.add(req.user_id,req.prompt,ans)
        return {"content_id":cid,"content":ans,"user_id":req.user_id}
    @app.get("/history/{user_id}")
    def hist(user_id:str): items=store.list_user(user_id); return {"user_id":user_id,"count":len(items),"items":items}
    @app.get("/content/{cid}")
    def gc(cid:str):
        item=store.get(cid)
        if item is None: raise HTTPException(404)
        return item
    return app

# — run integration tests —
c = TestClient(build_core(plan="free", process_fn=str.upper), raise_server_exceptions=False)

assert c.get("/health").json()["status"] == "ok"; print("\u2705 /health")
assert c.get("/plan").json()["plan"] == "free"; print("\u2705 /plan")
assert c.get("/metrics").status_code == 200; print("\u2705 /metrics")
assert len(c.get("/templates").json()["templates"]) > 0; print("\u2705 /templates")

r = c.post("/generate", json={"prompt":"hi","user_id":"u1"})
assert r.status_code == 200 and r.json()["content"] == "HI"
cid = r.json()["content_id"]; print("\u2705 /generate 200")

assert c.get("/history/u1").json()["count"] == 1; print("\u2705 /history")
assert c.get(f"/content/{cid}").status_code == 200; print("\u2705 /content/{id}")
assert c.get("/content/bad").status_code == 404; print("\u2705 /content/bad 404")

assert c.post("/generate", json={"prompt":"","user_id":"u"}).status_code == 422
print("\u2705 empty prompt 422")

c2 = TestClient(build_core(plan="free", process_fn=str.upper, initial_usage=5), raise_server_exceptions=False)
assert c2.post("/generate", json={"prompt":"x","user_id":"u"}).status_code == 429
print("\u2705 rate limit 429")

m = c.get("/metrics").json()
assert m["requests"] >= 5 and "error_rate" in m; print("\u2705 metrics correct")

print("\nSection 4 capstone complete! \U0001f389 shipped_app.py is ready to deploy.")
